# Introduction to LangChain

**Author:** Shinin Varongchayakul

**Date:** 30 Jan 2026

## 1. API Key

In [ ]:
# Import packages
from pathlib import Path
from dotenv import load_dotenv
import os

# Get .env file path
PROJECT_ROOT = Path.cwd().parents[2]
env_path = PROJECT_ROOT / ".env"

# Load variables from .env
load_dotenv(env_path, override=True)

# Get Gemini API key
gemini_api_key = os.getenv("GEMINI_API_KEY_01")

## 2. LLM

In [ ]:
# Import package
from langchain_google_genai import ChatGoogleGenerativeAI

# Create model instance
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.5,
    api_key=gemini_api_key
)

## 3. Prompt Template

In [ ]:
# Import package
from langchain_core.prompts import ChatPromptTemplate

# Define system prompt
system_prompt = """
You are an expert curator of mental models across science, philosophy, and applied reasoning.

Your task is to explain mental models clearly and accurately using a fixed schema.

If the origin of a model is unclear or debated, state that explicitly.

Do not invent historical sources. Be concise and concrete.
"""

# Define user prompt
user_prompt = "Explain the following mental model: {model_query}"

# Create prompt template
prompt = ChatPromptTemplate.from_messages(
    [
        # System prompt
        ("system", system_prompt.strip()),

        # User prompt
        ("human", user_prompt)
    ]
)

In [12]:
# Inspect prompt
prompt.format_messages(model_query="Pareto Pricinple")

[SystemMessage(content='You are an expert curator of mental models across science, philosophy, and applied reasoning.\n\nYour task is to explain mental models clearly and accurately using a fixed schema.\n\nIf the origin of a model is unclear or debated, state that explicitly.\n\nDo not invent historical sources. Be concise and concrete.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Explain the following mental model: Pareto Pricinple', additional_kwargs={}, response_metadata={})]

## 4. Output Format

In [ ]:
# Import packages
from pydantic import BaseModel, Field
from typing import List, Literal

# Define output structure
class MentalModel(BaseModel):

    # Mental model name
    model_name: str = Field(description="The commonly accepted name of the mental model")

    # Source/origin
    origin: str = Field(
        description="Where the model comes from (person, book, field, or cultural origin)"
    )

    # Brief description
    description: str = Field(
        description="A brief explanation of what the mental model is and why it matters"
    )

    # Examples
    examples: List[str] = Field(
        description="Concrete real-world examples illustrating the mental model"
    )

    # Cognitive load required to learn this model
    cognitive_load: Literal["low", "medium", "high"] = Field(
        description="How mentally demanding this model is to learn and apply"
    )

    # Tags
    tags: List[str] = Field(
        description="Short tags such as decision-making, systems thinking, learning, philosophy"
    )

In [14]:
# Add output structure to LLM
llm_with_structured_output = llm.with_structured_output(MentalModel)

## 5. Chain

In [15]:
# Build chain
chain = prompt | llm_with_structured_output

## 6. Single Run

In [16]:
# Run query
result = chain.invoke({"model_query": "Compound Interest"})

In [ ]:
# Import package
import json

# Load result
result_dict = result.model_dump()

# Print
print(json.dumps(result_dict, indent=4))

{
    "model_name": "Compound Interest",
    "origin": "Finance, mathematics. The concept was understood and applied in various forms by ancient civilizations, with formal mathematical treatment appearing in medieval Europe, notably in Luca Pacioli's 'Summa de arithmetica, geometria, proportioni et proportionalita' (1494).",
    "description": "Compound interest is the process where interest is earned not only on the initial principal but also on the accumulated interest from previous periods. It matters because it demonstrates the power of exponential growth, allowing small, consistent efforts or investments to yield significant results over time, both positively (wealth building) and negatively (debt accumulation).",
    "examples": [
        "An investment of $1,000 earning 5% annual interest will grow faster if the interest is reinvested each year, as the 5% is then calculated on a larger principal.",
        "Credit card debt can quickly spiral out of control because interest is c

## 7. Batch run

In [20]:
# Create list of mental models
mental_model_queries = [
    "First Principles Thinking",
    "Occam's Razor",
    "Confirmation Bias",
    "Map is Not the Territory",
    "42"
]

# Create batch inputs
batch_inputs = [{"model_query": query} for query in mental_model_queries]

# Run queries
results = chain.batch(batch_inputs)

# Instantiate collector
query_collector = [result.model_dump() for result in results]

In [21]:
# Instantiate counter
i = 1

# Loop through elements in collector
for result in query_collector:

    # Print result
    print(f"👉 Query {i}:")
    print(json.dumps(result, indent=4))
    print("\n")

    # Add 1 to counter
    i += 1

👉 Query 1:
{
    "model_name": "First Principles Thinking",
    "origin": "Ancient Greek philosophy (Aristotle), popularized in modern business by Elon Musk",
    "description": "First Principles Thinking is a way of breaking down complex problems into basic, fundamental truths and then building up from there. Instead of reasoning by analogy (doing things the way they've always been done or how others do them), it involves identifying the core components and underlying assumptions to create novel solutions or understanding. It matters because it fosters innovation, deeper understanding, and allows for original thought unconstrained by existing paradigms.",
    "examples": [
        "Elon Musk's approach to building rockets at SpaceX: instead of accepting the high cost of existing rockets, he broke down the cost of raw materials and manufacturing processes to find ways to drastically reduce expenses.",
        "A chef developing a new dish by understanding the fundamental properties of 